In [ ]:
# # For kaggle:
# !git clone https://github.com/maTiahK/CORT_SI.git
# %cd /kaggle/working/CORT_SI

# !python -m pip install -q --upgrade pip
# !python -m pip install -q -e .


# from cort_si import SI, SI_randj, gen_data
# print("CORT_SI import OK")

# Example 2: Pivot Uniformity Check for Adaptive CoRT-SI

This notebook evaluates whether selective p-values from `SI_randj(...)` look approximately uniform under a null target model.

In [ ]:
import random
import sys

import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

sys.path.append('..')

from cort_si import SI_randj, gen_data

## 1. Set Null Data-Generating Process

In [ ]:
p = 300
s = 5
nS = 100
nT = 50
true_beta = 0.0
num_info_aux = 1
num_uninfo_aux = 1
gamma = 0.05

lambda_sel = 0.25
lambda0 = 0.25
lambdak_list = [0.25] * (num_info_aux + num_uninfo_aux)
T = 3
max_iter = 100
seed = 0
verbose_preview = True
verbose_pivot = False

print(f'Feature dimension: {p}')
print(f'Sparsity level: {s}')
print(f'Source sample size: {nS}')
print(f'Target sample size: {nT}')
print(f'Number of auxiliary groups: {num_info_aux + num_uninfo_aux}')
print(f'max_iter: {max_iter}')
print('Truncation defaults: z_min=-20, z_max=20')
print(f'verbose_preview: {verbose_preview}')
print(f'verbose_pivot: {verbose_pivot}')

## 2. Run the Pivot Experiment

In [ ]:
def generate_null_instance(iteration_seed):
    np.random.seed(iteration_seed)
    random.seed(iteration_seed)
    return gen_data.generate_data(
        p=p,
        s=s,
        nS=nS,
        nT=nT,
        true_beta=true_beta,
        num_info_aux=num_info_aux,
        num_uninfo_aux=num_uninfo_aux,
        gamma=gamma,
    )


def preview_cort_run(seed=0, verbose=False):
    XS_list, YS_list, X0, Y0, _, SigmaS_list, Sigma0, _ = generate_null_instance(seed)
    return SI_randj(
        X0,
        Y0,
        XS_list,
        YS_list,
        lambda_sel=lambda_sel,
        lambda0=lambda0,
        lambdak_list=lambdak_list,
        SigmaS_list=SigmaS_list,
        Sigma0=Sigma0,
        T=T,
        verbose=verbose,
    )


def run_cort_pivot(max_iter=100, seed=0, verbose=False):
    p_sel_list = []

    for iteration in range(max_iter):
        XS_list, YS_list, X0, Y0, _, SigmaS_list, Sigma0, _ = generate_null_instance(seed + iteration)

        p_value_rand = SI_randj(
            X0,
            Y0,
            XS_list,
            YS_list,
            lambda_sel=lambda_sel,
            lambda0=lambda0,
            lambdak_list=lambdak_list,
            SigmaS_list=SigmaS_list,
            Sigma0=Sigma0,
            T=T,
            verbose=verbose,
        )

        if p_value_rand is not None and np.isfinite(p_value_rand):
            p_sel_list.append(p_value_rand)

        if (iteration + 1) % 20 == 0:
            print(f'Iteration {iteration + 1}/{max_iter}')

    return p_sel_list


preview_pvalue = preview_cort_run(seed=seed, verbose=verbose_preview)
print(f'Preview p-value: {preview_pvalue}')

pivot_pvalues = run_cort_pivot(max_iter=max_iter, seed=seed, verbose=verbose_pivot)

## 3. Analyze Uniformity

In [ ]:
print(f'Collected {len(pivot_pvalues)} valid p-values out of {max_iter} simulations')

if len(pivot_pvalues) == 0:
    print('No valid p-values were produced. Increase max_iter or relax the demo settings.')
else:
    pivot_array = np.array(pivot_pvalues)
    ks_stat, ks_pvalue = stats.kstest(pivot_array, 'uniform')
    prop_below_005 = np.mean(pivot_array <= 0.05)

    print(f'KS statistic: {ks_stat:.4f}')
    print(f'KS p-value: {ks_pvalue:.4f}')
    print(f'Fraction below 0.05: {prop_below_005:.4f}')

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(pivot_array, bins=np.linspace(0, 1, 11), density=True, edgecolor='black', alpha=0.8)
    axes[0].axhline(1.0, color='crimson', linestyle='--', label='Uniform density')
    axes[0].set_title('Histogram of selective p-values')
    axes[0].set_xlabel('p-value')
    axes[0].set_ylabel('Density')
    axes[0].legend()

    sorted_p = np.sort(pivot_array)
    ecdf = np.arange(1, len(sorted_p) + 1) / len(sorted_p)
    axes[1].plot(sorted_p, ecdf, marker='o', linestyle='', label='Empirical CDF')
    axes[1].plot([0, 1], [0, 1], color='crimson', linestyle='--', label='Uniform CDF')
    axes[1].set_title('Empirical CDF vs uniform CDF')
    axes[1].set_xlabel('p-value')
    axes[1].set_ylabel('CDF')
    axes[1].legend()

    plt.tight_layout()
    plt.show()